# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
!pip -q install datasets huggingface_hub duckdb pandas pyarrow

In [20]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

In [21]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [22]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [23]:
con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [24]:
df = con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 100
""").df()

df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [25]:
con.sql(f"""
SELECT
    MIN(month) AS first_month,
    MAX(month) AS last_month,
    COUNT(DISTINCT month) AS total_months
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

┌─────────────┬────────────┬──────────────┐
│ first_month │ last_month │ total_months │
│   varchar   │  varchar   │    int64     │
├─────────────┼────────────┼──────────────┤
│ 2025-01     │ 2026-06    │           18 │
└─────────────┴────────────┴──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis + Time Window

Each row represents the daily performance of one content page (`content_hash_id`) for one client (`client_hash_id`) on a specific reporting date (`report_date`).

The available data spans from **January 2025** to **June 2026**, covering **18 months** of observations.

For exploratory analysis and development, I will use a mid-period month (such as **2026-03**) to reduce the risk of using information from the final evaluation period.

In [26]:
con.sql(f"""
SELECT
    MIN(month) AS first_month,
    MAX(month) AS last_month,
    COUNT(DISTINCT month) AS total_months
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

┌─────────────┬────────────┬──────────────┐
│ first_month │ last_month │ total_months │
│   varchar   │  varchar   │    int64     │
├─────────────┼────────────┼──────────────┤
│ 2025-01     │ 2026-06    │           18 │
└─────────────┴────────────┴──────────────┘

In [27]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '-' || content_hash_id || '-' || CAST(report_date AS VARCHAR)) AS unique_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────┐
│ total_rows │ unique_rows │
│   int64    │    int64    │
├────────────┼─────────────┤
│   78835655 │    78829265 │
└────────────┴─────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Field Classification

### Features
These fields describe search performance and user engagement before a prediction is made.

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

### Label
The prediction target is not defined in this notebook. This assignment focuses on documenting the data contract and identifying candidate features. The target variable will be defined during the modeling stage.

### Context
These fields identify the observation and provide useful metadata.

- client_hash_id
- content_hash_id
- report_date
- month
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

### Excluded
The following fields are not used in this exercise because they are outside the current scope of the data contract and may not be consistently available for every observation.

- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [28]:
con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│   78835655 │
└────────────┘

In [29]:
con.sql(f"""
SELECT
MIN(month) AS first_month,
MAX(month) AS last_month
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

┌─────────────┬────────────┐
│ first_month │ last_month │
│   varchar   │  varchar   │
├─────────────┼────────────┤
│ 2025-01     │ 2026-06    │
└─────────────┴────────────┘

In [ ]:
con.sql(f"""
SELECT
SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks,
SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS missing_sessions
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.sql(f"""
SELECT
gsc_data_available,
ga4_data_available,
COUNT(*) AS rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
GROUP BY gsc_data_available, ga4_data_available
ORDER BY rows DESC
""")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- The available data covers the period from January 2025 to June 2026 only.
- Some early records may contain Google Search Console data without complete GA4 metrics.
- Client and content identifiers are anonymized using hash values, so the original client names and URLs cannot be examined.
- The dataset measures observed search and engagement metrics but does not directly capture content quality, business value, or user satisfaction.
- This dataset should be used to support decision-making rather than establish cause-and-effect relationships.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.